# DTB oscillatory multi-well potential-game experiment

This controlled sweep tests whether a fixed neural tangent representation becomes less accurate as the game vector field becomes more oscillatory. It compares $\omega\in\{2\pi,4\pi,8\pi,16\pi\}$ for the decoupled baseline $\gamma=0$ and coupled game $\gamma=0.2$. The hypothesis is tested from the data; it is not assumed beforehand.

## Block 1 -- Clone the Game-DTB branch and install dependencies

In [ ]:
import pathlib, shutil, subprocess, sys

REPO_URL = "https://github.com/sun-mengwei/dtb-colab-experiments.git"
BRANCH = "codex/game-dynamics-dtb"
REPO_DIR = pathlib.Path("/content/dtb-colab-experiments")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
%cd /content/dtb-colab-experiments/dtb_game_dynamics_unnormalized
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest"], check=True)

## Block 2 -- Game definition and equilibrium convention

The common potential and ascent field are

$$\Phi=-\frac{\lambda}{2}(x_1^2+x_2^2)-\frac{\gamma}{2}(x_1-x_2)^2+\frac{\varepsilon}{\omega}[\cos(\omega x_1)+\cos(\omega x_2)],$$

$$b_1=-\lambda x_1-\gamma(x_1-x_2)-\varepsilon\sin(\omega x_1),\qquad b_2=-\lambda x_2-\gamma(x_2-x_1)-\varepsilon\sin(\omega x_2).$$

Equilibria are numerical roots of $b(x)=0$ in $[-1,1]^2$. Red circles are linearly stable and gold `X` markers are unstable. Root enumeration is numerical rather than a proof of completeness.

## Block 3 -- Changeable controlled-experiment setup

Use `SMOKE=True` first. Then set it to `False` for the requested full sweep. Within one run of the sweep, only `OMEGA_MULTIPLIERS` and `GAMMAS` vary; the script verifies that initial particles and neural initialization remain identical.

In [ ]:
# Run mode
SMOKE = True                    # True: tiny correctness run; False: use values below
OUTPUT_ROOT = "outputs/oscillatory_potential_game"

# Game sweep
OMEGA_MULTIPLIERS = [2, 4, 8, 16]  # means omega = multiplier*pi
GAMMAS = [0.0, 0.2]
LAMBDA_VALUE = 0.5
EPSILON = 0.5

# Shared particles and physical time grid
PARTICLES = 2000
STEPS = 200
STEP_SIZE = 0.005               # T = 1
SEED = 2026
MODEL_SEED = 91

# Initial NN architecture and tangent bundle
ARCHITECTURE = "mlp"          # mlp, mmnn, or node
WIDTH = 32
DEPTH = 4
MMNN_RANK = 8
BASIS_SIZE = 128
SVD_RTOL = 1e-3
ACTIVATION = "tanh"

# Existing DTB periodic reset: zero gives the fixed-representation baseline
REFIT_INTERVAL = 0
REFIT_OPTIMIZER_STEPS = 2000

# Reference and basin diagnostics
REFERENCE_SUBSTEPS = 20          # RK4 substeps per DTB step
BASIN_TOLERANCE = 0.08
MAX_UNASSIGNED_FRACTION = 0.05
DEVICE = "auto"
DTYPE = "float32"

## Block 4 -- Run the sweep

The high-accuracy RK4 reference and DTB start from exactly the same particles. The reference is recomputed with twice as many substeps to report a final self-convergence error. This does not change any DTB parameter.

In [ ]:
command = [
    sys.executable, "run_oscillatory_potential_game.py",
    "--omega-multipliers", *map(str, OMEGA_MULTIPLIERS),
    "--gammas", *map(str, GAMMAS),
    "--lambda-value", str(LAMBDA_VALUE),
    "--epsilon", str(EPSILON),
    "--particles", str(PARTICLES),
    "--steps", str(STEPS),
    "--step-size", str(STEP_SIZE),
    "--seed", str(SEED),
    "--model-seed", str(MODEL_SEED),
    "--architecture", ARCHITECTURE,
    "--width", str(WIDTH),
    "--depth", str(DEPTH),
    "--rank", str(MMNN_RANK),
    "--basis-size", str(BASIS_SIZE),
    "--svd-rtol", str(SVD_RTOL),
    "--activation", ACTIVATION,
    "--refit-interval", str(REFIT_INTERVAL),
    "--refit-optimizer-steps", str(REFIT_OPTIMIZER_STEPS),
    "--reference-substeps", str(REFERENCE_SUBSTEPS),
    "--basin-tolerance", str(BASIN_TOLERANCE),
    "--max-unassigned-fraction", str(MAX_UNASSIGNED_FRACTION),
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--output-root", OUTPUT_ROOT,
]
if SMOKE:
    command.append("--smoke")
subprocess.run(command, check=True)

## Block 5 -- Cross-frequency plots and summary table

Do not draw scientific conclusions from the smoke preset. For the full sweep, compare frequency trends in projection residual, trajectory RMSE, basin error, retained rank, and condition number. Basin error is reported as unavailable when too many terminal particles cannot be assigned reliably.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, display

root = pathlib.Path(OUTPUT_ROOT)
summary = pd.read_csv(root / "summary.csv")
display(summary)
for filename in [
    "gamma_0_frequency_comparison.png",
    "gamma_0p2_frequency_comparison.png",
    "final_error_vs_frequency.png",
]:
    path = root / filename
    if path.exists():
        print(filename)
        display(Image(str(path)))

## Block 6 -- Inspect one case in detail

Change the two labels below to inspect another run. The diagnostic figure contains projection residual, absolute/relative trajectory error, velocity RMSE, retained rank, retained condition number, and reference/DTB mean potential.

In [ ]:
INSPECT_GAMMA_LABEL = "0p2"
INSPECT_OMEGA_LABEL = "16"
case = root / f"gamma_{INSPECT_GAMMA_LABEL}" / f"omega_{INSPECT_OMEGA_LABEL}pi"
for filename in [
    "potential_vector_field.png",
    "particle_comparison.png",
    "diagnostics.png",
]:
    display(Image(str(case / filename)))
display(pd.read_csv(case / "equilibria.csv").head(30))

## Block 7 -- Quantify which diagnostic tracks trajectory error

These correlations summarize eight cases only, so treat them as descriptive rather than inferential. A positive residual/error correlation supports the representation-capacity explanation; a stronger condition/error or rank/error relationship points more toward the truncated-SVD solve.

In [ ]:
columns = [
    "final_trajectory_rmse",
    "mean_projection_residual",
    "mean_retained_rank",
    "max_retained_condition",
    "max_alpha_norm",
]
display(summary[columns].corr().loc[["final_trajectory_rmse"]])
display(summary.groupby("gamma")[["mean_projection_residual", "final_trajectory_rmse", "mean_retained_rank", "max_retained_condition"]].corr())

## Block 8 -- Save outputs to Google Drive (optional)

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = pathlib.Path("/content/drive/MyDrive/dtb_oscillatory_potential_game")
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(root, destination)
    print("saved to", destination)